In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# совместимый версии
!pip install torchtext=='0.18.0' torch=='2.3.0' torchdata=='0.9.0' portalocker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176

In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torchtext import datasets
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [4]:
train_data = datasets.IMDB(split='train')
test_data = datasets.IMDB(split='test')

In [5]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [6]:
tokenizer = get_tokenizer("basic_english")

In [7]:
# проверяем какой столбец первым приходит(класс или данные)
for i in list(train_data)[:5]: # берём только 5 первых
    print(i)

(1, "Heavy-handed moralism. Writers using characters as mouthpieces to speak for themselves. Predictable, plodding plot points (say that five times fast). A child's imitation of Britney Spears. This film has all the earmarks of a Lifetime Special reject.<br /><br />I honestly believe that Jesus Nebot and Julia Montejo set out to create a thought-provoking, emotional film on a tough subject, exploring the idea that things are not always black and white, that one who is a criminal by definition is not necessarily a bad human being, and that there can be extenuating circumstances, especially when one puts the well-being of a child first. However, their earnestness ends up being channeled into preachy dialogue and trite situations planted to move the plot along. The decent production values and interesting use of documentary-style camera footage are not enough to accomplish their aim when the script and the acting fall flat.<br /><br />Logic is often compromised for the sake of creating te

In [8]:
def text_to_token(data):
    for label, text in data:
        yield tokenizer(text)

In [9]:
vocab = build_vocab_from_iterator(text_to_token(train_data), specials=["<unk>"])

In [10]:
vocab.set_default_index(vocab["<unk>"])

In [11]:
def change_text(x):
    return [vocab[i] for i in tokenizer(x)]

change_label = lambda label: 1 if label == 2 else 0

In [12]:
def collate_batch(batch):
    labels, texts = [], []
    for label, text in batch:
        labels.append(change_label(label))
        texts.append(torch.tensor(change_text(text), dtype=torch.int64))
    labels = torch.tensor(labels, dtype=torch.int64)
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    return texts, labels

In [13]:
train_dataloader = DataLoader(list(train_data), batch_size=32, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(list(test_data), batch_size=32, collate_fn=collate_batch)

In [14]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
model = SentimentModel(len(vocab)).to(device)

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [18]:
for epoch in range(15):
    model.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model(texts)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Эпоха: {epoch + 1} - Потери: {round(total_loss, 2)}')

Эпоха: 1 - Потери: 543.0
Эпоха: 2 - Потери: 541.61
Эпоха: 3 - Потери: 540.79
Эпоха: 4 - Потери: 539.15
Эпоха: 5 - Потери: 536.23
Эпоха: 6 - Потери: 534.6
Эпоха: 7 - Потери: 532.73
Эпоха: 8 - Потери: 533.03
Эпоха: 9 - Потери: 499.91
Эпоха: 10 - Потери: 409.34
Эпоха: 11 - Потери: 310.1
Эпоха: 12 - Потери: 232.57
Эпоха: 13 - Потери: 176.09
Эпоха: 14 - Потери: 134.56
Эпоха: 15 - Потери: 141.02


In [19]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 85.19%


In [20]:
from google.colab import files

torch.save(vocab, 'vocab_IMDB_DatasetOf_50K_MovieReviews.pth')
torch.save(model.state_dict(), 'model_IMDB_DatasetOf_50K_MovieReviews.pth')

files.download('vocab_IMDB_DatasetOf_50K_MovieReviews.pth')
files.download('model_IMDB_DatasetOf_50K_MovieReviews.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>